In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# --- 1. LOCAL SETUP ---
CHECKPOINT_PATH = r"C:\Users\park2\Desktop\ME592_HW2\robotic-grasping\sam3.pt"
bpe_path = "bpe_simple_vocab_16e6.txt.gz"
image_dir = r"C:\Users\park2\Desktop\ME592_HW2\robotic-grasping\6_test_images"  

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading SAM 3 onto {device}...")

model = build_sam3_image_model(checkpoint_path=CHECKPOINT_PATH, bpe_path=bpe_path).to(device)
processor = Sam3Processor(model)

# --- 2. YOUR HAND-WRITTEN COORDINATES ---
# Converted your 4 corners into [x_min, y_min, x_max, y_max] bounding boxes
data = {
    "pcd0800r.png": [271, 261, 323, 343],
    "pcd0801r.png": [227, 191, 350, 306],
    "pcd0802r.png": [219, 174, 351, 290],
    "pcd0803r.png": [211, 192, 363, 309],
    "pcd0804r.png": [205, 153, 381, 341],
    "pcd0805r.png": [137, 151, 396, 362]
}

# --- 3. THE EVALUATION LOOP ---
for filename, box in data.items():
    img_path = os.path.join(image_dir, filename)
    if not os.path.exists(img_path):
        print(f"Skipping {filename} - File not found!")
        continue

    print(f"\nProcessing {filename}...")
    image_pil = Image.open(img_path).convert("RGB")
    inference_state = processor.set_image(image_pil)
    
    # --- PROMPT A: 3 FOREGROUND POINTS ---
    # Calculate 3 points inside the bounding box (center, upper-middle, lower-middle)
    cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
    height = box[3] - box[1]
    
    # Format: [[x1, y1], [x2, y2], [x3, y3]]
    point_coords = [[cx, cy], [cx, cy - height*0.25], [cx, cy + height*0.25]]
    # Labels: 1 means foreground (positive click)
    point_labels = [1, 1, 1] 

    output_points = processor.set_point_prompt(
        state=inference_state, 
        points=point_coords, 
        labels=point_labels
    )
    mask_points = output_points["masks"][0].cpu().numpy().squeeze() if torch.is_tensor(output_points["masks"][0]) else output_points["masks"][0].squeeze()
    score_points = output_points["scores"][0].item() if torch.is_tensor(output_points["scores"][0]) else output_points["scores"][0]

    # --- PROMPT B: BOUNDING BOX ---
    # We have to reset the state or it will combine the points AND the box
    inference_state = processor.set_image(image_pil) 
    
    output_box = processor.set_box_prompt(
        state=inference_state, 
        boxes=[box]
    )
    mask_box = output_box["masks"][0].cpu().numpy().squeeze() if torch.is_tensor(output_box["masks"][0]) else output_box["masks"][0].squeeze()
    score_box = output_box["scores"][0].item() if torch.is_tensor(output_box["scores"][0]) else output_box["scores"][0]

    # --- 4. SIDE-BY-SIDE VISUALIZATION ---
    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    fig.suptitle(f"File: {filename}", fontsize=20, fontweight='bold')

    # Plot A: Points
    axes[0].imshow(image_pil)
    axes[0].imshow(mask_points, alpha=0.5, cmap='autumn')
    for px, py in point_coords:
        axes[0].plot(px, py, 'o', color='green', markersize=8, markeredgecolor='white', markeredgewidth=2)
    axes[0].set_title(f"3 Point Prompts\nConfidence Score: {score_points:.2f}", fontsize=14)
    axes[0].axis('off')

    # Plot B: Box
    axes[1].imshow(image_pil)
    axes[1].imshow(mask_box, alpha=0.5, cmap='winter')
    rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], edgecolor='blue', facecolor='none', lw=3)
    axes[1].add_patch(rect)
    axes[1].set_title(f"Box Prompt\nConfidence Score: {score_box:.2f}", fontsize=14)
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

Loading SAM 3 onto cuda...
Skipping pcd0800r.png - File not found!
Skipping pcd0801r.png - File not found!
Skipping pcd0802r.png - File not found!
Skipping pcd0803r.png - File not found!
Skipping pcd0804r.png - File not found!
Skipping pcd0805r.png - File not found!
